<a href="https://colab.research.google.com/github/Adarsha2004/finetuning/blob/main/gemma_cft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q unsloth
%pip install -q datasets
%pip install mlx-tune


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import re
import random
import math

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0
CUDA available: False


In [3]:
from mlx_tune import FastVisionModel

import mlx.core as mx
import mlx.utils

BASE_MODEL = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 1024
SEED = 42

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

FastVisionModel.for_inference(model)

print(f"\n[OK] Model loaded: {BASE_MODEL}")
num_params = sum(p.size for _, p in mlx.utils.tree_flatten(model.parameters()))
print(f"Parameters: {num_params:,}")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W0429 00:21:04.006000 19838 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading VLM: unsloth/gemma-4-E2B-it


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 93503.59it/s]
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/transformers/audio_utils.py:538: UserWarning: At least one mel filter has all zero values. The value for `num_mel_filters` (128) may be set too high. Or, the value for `num_frequency_bins` (257) may be set too low.
  warnings.warn(
Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 87642.17it/s]


[OK] Model loaded: unsloth/gemma-4-E2B-it
Parameters: 5,123,178,979


In [4]:
def compute_perplexity(model, tokenizer, texts, max_length=512):
    """
    Calculates perplexity using a neutral assistant role.
    This avoids forcing the model into a 'summary' task and measures
    raw domain likelihood more accurately.
    """
    total_loss = 0
    total_tokens = 0

    model.eval()
    for text in texts:
        # Use a neutral prompt to establish domain context without a task
        messages = [
            {"role": "user", "content": [{"type": "text", "text": "Provide medical textbook information:"}]},
            {"role": "assistant", "content": [{"type": "text", "text": text}]}
        ]

        full_input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
            return_tensors="pt"
        ).to(model.device)

        prompt_input_ids = tokenizer.apply_chat_template(
            messages[:1],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

        prompt_len = prompt_input_ids.shape[1]

        labels = full_input_ids.clone()
        labels[:, :prompt_len] = -100

        with torch.no_grad():
            outputs = model(input_ids=full_input_ids, labels=labels)

        num_tokens = (labels != -100).sum().item()
        if num_tokens > 0:
            total_loss += outputs.loss.item() * num_tokens
            total_tokens += num_tokens

    if total_tokens == 0: return float('inf')

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)

In [5]:
# Re-evaluating general text with neutral prompt
general_texts = [
    "The weather was beautiful that morning as the children walked to school through the park.",
    "Scientists have discovered a new species of butterfly in the Amazon rainforest.",
    "The recipe calls for two cups of flour, one egg, and a tablespoon of butter.",
    "The basketball game went into overtime after a dramatic three-point shot.",
    "She opened the book and began reading the first chapter aloud to her students.",
]

ppl = compute_perplexity(model, tokenizer, general_texts)
print(f"General Perplexity (Neutral Prompt): {ppl:.2f}")

General Perplexity (Neutral Prompt): 232.13


In [6]:
sample_medical = [
    "The patient presented with acute chest pain radiating to the left arm, accompanied by diaphoresis and shortness of breath, suggestive of myocardial infarction.",
    "Hemoglobin levels were recorded at 8.2 g/dL, indicating moderate anemia, and iron supplementation was initiated.",
    "MRI findings revealed a lesion in the temporal lobe consistent with early-stage glioma.",
    "The individual reported persistent hyperglycemia with fasting blood glucose levels above 140 mg/dL, indicating uncontrolled diabetes mellitus.",
    "Antibiotic therapy with amoxicillin-clavulanate was prescribed for bacterial sinusitis following clinical evaluation.",
]

medical_ppl = compute_perplexity(model, tokenizer, sample_medical)

print(f"Medical Perplexity (Neutral Prompt): {medical_ppl:.1f}")
print("(This is our more accurate baseline for CPT)")

Medical Perplexity (Neutral Prompt): 139.7
(This is our more accurate baseline for CPT)


In [5]:
from huggingface_hub import hf_hub_download
import pandas as pd

configs = [
    "Pathology_Robbins",
    "Physiology_Levy",
    "Pharmacology_Katzung",
    "Neurology_Adams",
    "Pediatrics_Nelson"
]

NUM_TRAIN = 5000
NUM_VAL = 500
TARGET_TOTAL = NUM_TRAIN + NUM_VAL

all_texts = []

print(f"[...] Collecting {TARGET_TOTAL} samples...")

# -----------------------------
# LOAD DATA (direct parquet — bypasses dill/pickle issue on Python 3.14)
# -----------------------------
for cfg in configs:
    print(f"\n[+] Loading: {cfg}")

    file_path = hf_hub_download(
        repo_id="zxvix/MedicalTextbook",
        filename=f"{cfg}/train-00000-of-00001.parquet",
        repo_type="dataset"
    )

    df = pd.read_parquet(file_path)

    for _, row in df.iterrows():
        text = row.get("text", "")

        if text and len(str(text).split()) > 100:
            all_texts.append(str(text))

        if len(all_texts) >= TARGET_TOTAL:
            break

    if len(all_texts) >= TARGET_TOTAL:
        break

print(f"\n[OK] Total collected: {len(all_texts)}")


[...] Collecting 5500 samples...

[+] Loading: Pathology_Robbins

[+] Loading: Physiology_Levy

[+] Loading: Pharmacology_Katzung

[+] Loading: Neurology_Adams

[OK] Total collected: 5500


In [7]:
sample = all_texts[0]
words = sample.split()
print(f"Sample filing length: {len(words)} words")
print(f"\nFirst 200 words:")
print(" ".join(words[:200]))
print("\n...")
print(f"\nLast 100 words:")
print(" ".join(words[-100:]))

Sample filing length: 498 words

First 200 words:
Plasma Membrane: Protection and Nutrient Acquisition Biosynthetic Machinery: Endoplasmic Reticulum and Golgi Apparatus Waste Disposal: Lysosomes and Proteasomes Modular Signaling Proteins, Hubs, and Components of the Extracellular Matrix Proliferation and the Cell Cycle Pathology literally translates to the study of suffering (Greek pathos = suffering, logos = study); as applied to modern medicine, it is the study of disease. Virchow was certainly correct in asserting that disease originates at the cellular level, but we now realize that cellular disturbances arise from alterations in molecules (genes, proteins, and others) that influence the survival and behavior of cells. Thus, the foundation of modern pathology is understanding the cellular and molecular abnormalities that give rise to diseases. It is helpful to consider these abnormalities in the context of normal cellular structure and function, which is the theme of this introduct

In [8]:
# -----------------------------
# CLEAN TEXT
# -----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)

    # Fix merged words
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    text = re.sub(r"([a-zA-Z])(\d)", r"\1 \2", text)
    text = re.sub(r"(\d)([a-zA-Z])", r"\1 \2", text)

    # Fix commas spacing
    text = re.sub(r",([a-zA-Z])", r", \1", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

cleaned_texts = [clean_text(t) for t in all_texts]


In [9]:
# -----------------------------
# SHUFFLE + SPLIT (IMPORTANT)
# -----------------------------
random.shuffle(cleaned_texts)

train_texts = cleaned_texts[:NUM_TRAIN]
val_texts   = cleaned_texts[NUM_TRAIN:NUM_TRAIN + NUM_VAL]

print(f"\nTrain filings: {len(train_texts)}")
print(f"Val filings:   {len(val_texts)}")


Train filings: 5000
Val filings:   500


In [10]:
# -----------------------------
# CHUNKING
# -----------------------------
def chunk_texts(texts, chunk_size=512, overlap=0.1):
    step = int(chunk_size * (1 - overlap))
    all_chunks = []

    for text in texts:
        words = text.split()
        for i in range(0, len(words), step):
            chunk = words[i:i + chunk_size]
            if len(chunk) > 50:
                all_chunks.append(" ".join(chunk))

    return all_chunks

train_chunks = chunk_texts(train_texts, chunk_size=512, overlap=0.1)
val_chunks   = chunk_texts(val_texts, chunk_size=512, overlap=0.1)

print(f"\nTrain chunks: {len(train_chunks)}")
print(f"Val chunks:   {len(val_chunks)}")


Train chunks: 7249
Val chunks:   734


In [11]:
# -----------------------------
# SAMPLE OUTPUT
# -----------------------------
sample = train_chunks[0].split()

print("\nSample chunk (first 100 words):")
print(" ".join(sample[:100]))

print("\n...")
print("\nLast 50 words:")
print(" ".join(sample[-50:]))


Sample chunk (first 100 words):
Neurology 41:634, 1991. Ribot TH: Diseases of Memory: An Essay in Positive Psychology. New York, Appleton, 1882. Rowan AJ, Protass LM: Transient global amnesia: Clinical and electroencephalographic findings in 10 cases. Neurology 29:869, 1979. Sander D, Winbeck K, Eigen T, et al: Disturbance of venous flow patterns in patients with transient global amnesia. Lancet 356:1982, 2000. Schacter DL: Searching for Memory. New York, Basic Books, 1996. Schneider LS, Dagerman KS, Insel P. Risk of death with atypical antipsychotic drug treatment for dementia: Meta-analysis of randomized placebo-controlled trials. JAMA 294:1934, 2005. Schreiber SJ, Doepp F, Klingebiel R, Valdueza JM: Internal jugular

...

Last 50 words:
Stanford University Press, 1959. Thompson PM, Cannon TD, Narr KL, et al: Genetic influences on brain structure. Nat Neurosci 4:1253, 2001. Thurstone LL: The Vectors of the Mind. Chicago, University of Chicago Press, 1953. Tierney MC, Snow WG, Reid D

In [12]:
# ─── Fix Python 3.14 + datasets/dill pickle incompatibility ───
import sys
if sys.version_info >= (3, 14):
    import pickle
    import datasets.utils._dill as _datasets_dill

    def _fixed_batch_setitems(self, items, obj=None):
        if self._legacy_no_dict_keys_sorting:
            return pickle._Pickler._batch_setitems(self, items, obj)
        try:
            items = sorted(items)
        except Exception:
            from datasets.fingerprint import Hasher
            items = sorted(items, key=lambda x: Hasher.hash(x[0]))
        pickle._Pickler._batch_setitems(self, items, obj)

    _datasets_dill.Pickler._batch_setitems = _fixed_batch_setitems
    print("[OK] Patched datasets Pickler for Python 3.14")


[OK] Patched datasets Pickler for Python 3.14


In [13]:
# Convert to HuggingFace Dataset format
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_chunks})
val_dataset = Dataset.from_dict({"text": val_chunks})

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")

Train dataset: Dataset({
    features: ['text'],
    num_rows: 7249
})
Val dataset:   Dataset({
    features: ['text'],
    num_rows: 734
})


In [14]:
# IMPORTANT: Measure base model perplexity on the ACTUAL val chunks
# We need this before we reload the model for training
val_sample = val_chunks[:50]
base_ppl_val = compute_perplexity(model, tokenizer, val_sample)
print(f"Base model perplexity on medical validation chunks: {base_ppl_val:.1f}")
print("(We'll compare this to the CPT model later)")

Base model perplexity on medical validation chunks: 50.6
(We'll compare this to the CPT model later)


In [14]:
import gc
import time
import torch

# More robust memory recovery
try:
    del model
    del tokenizer
    del trainer
except NameError:
    pass

for _ in range(3):
    gc.collect()
    torch.cuda.empty_cache()

time.sleep(5) # Extended pause for VRAM stabilization

from mlx_tune import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=BASE_MODEL,
    load_in_4bit=True,
)

tokenizer = tokenizer.tokenizer

# --- ADD PEFT ADAPTERS ---
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # ← freeze vision
    finetune_language_layers=True,     # ← train only language
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",
                    "embed_tokens", "lm_head"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=True,
    loftq_config=None,
)

print(f"[OK] Model successfully reloaded and adapters attached.")

Loading VLM: unsloth/gemma-4-E2B-it


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 159565.91it/s]
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/transformers/audio_utils.py:538: UserWarning: At least one mel filter has all zero values. The value for `num_mel_filters` (128) may be set too high. Or, the value for `num_frequency_bins` (257) may be set too low.
  warnings.warn(
Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 16748.50it/s]


#trainable params: 25.337856 M || all params: 4647.449856 M || trainable%: 0.545%
LoRA configured: r=16, alpha=32, modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'embed_tokens', 'lm_head']
[OK] Model successfully reloaded and adapters attached.


In [15]:
from mlx_tune import CPTTrainer, CPTConfig

training_args = CPTConfig(
    output_dir="./cpt_medical",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    embedding_learning_rate=2e-5,
    include_embeddings=False,            # quantized model can't train these
    warmup_steps=20,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_seq_length=512,                  # ← was "max_length" — wrong param name
    dataset_text_field="text",
    logging_steps=10,
    save_steps=200,
    max_steps=1000,                      # ← cap iterations
    grad_checkpoint=True,
)

trainer = CPTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print("[OK] Trainer configured for 16GB Mac")


CPT Trainer initialized:
  Output dir: cpt_medical
  Learning rate: 0.0002
  Embedding learning rate: 2e-05
  Decoupled LR: True (scale=0.1000)
  Include embeddings: False
  Iterations: 1000
  Batch size: 1
  Dataset text field: text
[OK] Trainer configured for 16GB Mac


In [ ]:
# TRAIN!
# On a T4 GPU with 135M model + 80 filings + 2 epochs, ~8-12 minutes
print("=" * 70)
print("TRAINING STARTED")
print("=" * 70)
print("Watch the eval_loss -- that's your north star.")
print("It should decrease and then plateau.")
print()

trainer_stats = trainer.train()

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

TRAINING STARTED
Watch the eval_loss -- that's your north star.
It should decrease and then plateau.

Starting Continual Pretraining (CPT)

Trainable parameters: 26,517,504 / 5,148,516,835 (0.52%)
Preparing CPT training data (raw text)...
  Prepared 7249 training samples -> cpt_medical/train.jsonl
  Prepared validation set

[Using Decoupled LR Training]
  Main LR: 0.0002
  Embedding LR: 2e-05
  Gradient scale: 0.1000
Loaded 7249 training samples

Starting training for 1000 iterations...
  Step 10/1000 | Loss: 10.7772
  Step 20/1000 | Loss: 6.7362
  Step 30/1000 | Loss: 6.8890
  Step 40/1000 | Loss: 6.4658
  Step 50/1000 | Loss: 6.2074
  Step 60/1000 | Loss: 6.1980
  Step 70/1000 | Loss: 5.8708
  Step 80/1000 | Loss: 6.1900
  Step 90/1000 | Loss: 5.9424
  Step 100/1000 | Loss: 5.9645
  Step 110/1000 | Loss: 6.0112
  Step 120/1000 | Loss: 5.6989
  Step 130/1000 | Loss: 5.8212
  Step 140/1000 | Loss: 6.1337
  Step 150/1000 | Loss: 5.8367
  Step 160/1000 | Loss: 5.9361
  Step 170/1000 | Lo

AttributeError: 'dict' object has no attribute 'training_loss'

In [20]:
import mlx.core as mx
import mlx.nn as nn
import math

def compute_perplexity_mlx(model, processor, texts, max_length=512):
    """
    Compute perplexity using VLM model with proper processor tokenization.
    """
    actual_model = model.model if hasattr(model, 'model') else model
    # Get the tokenizer from processor
    tok = processor.tokenizer if hasattr(processor, 'tokenizer') else processor
    
    total_loss = 0.0
    total_tokens = 0

    for text in texts:
        # Use chat template for proper VLM input formatting
        messages = [
            {"role": "user", "content": [{"type": "text", "text": "Continue this text:"}]},
            {"role": "assistant", "content": [{"type": "text", "text": text}]},
        ]
        
        # Tokenize through processor
        prompt_text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        tokens = tok.encode(prompt_text)
        
        # Get prompt-only tokens to mask the prompt from loss
        prompt_msgs = [messages[0]]
        prompt_text_only = tok.apply_chat_template(
            prompt_msgs, tokenize=False, add_generation_prompt=True
        )
        prompt_tokens = tok.encode(prompt_text_only)
        prompt_len = len(prompt_tokens)
        
        if len(tokens) > max_length:
            tokens = tokens[:max_length]
        if len(tokens) < 2:
            continue

        input_ids = mx.array([tokens])

        output = actual_model(input_ids)
        logits = output.logits if hasattr(output, 'logits') else output
        logits = logits.astype(mx.float32)

        shift_logits = logits[:, :-1, :]
        shift_labels = input_ids[:, 1:]
        loss_per_token = nn.losses.cross_entropy(shift_logits, shift_labels, reduction='none')

        # Only count loss on the RESPONSE tokens (after prompt)
        if prompt_len - 1 < loss_per_token.shape[1]:
            response_loss = loss_per_token[0, prompt_len - 1:]
            total_loss += response_loss.sum().item()
            total_tokens += response_loss.size

    if total_tokens == 0:
        return float('inf')
    return math.exp(total_loss / total_tokens)


In [21]:
# TRAINED model (already loaded with LoRA)
from mlx_tune import FastVisionModel
FastVisionModel.for_inference(model)

# Use the original processor (not just tokenizer)
trained_med = compute_perplexity_mlx(model, tokenizer, medical_texts)
trained_gen = compute_perplexity_mlx(model, tokenizer, general_texts)
print(f"TRAINED — Medical: {trained_med:.2f}, General: {trained_gen:.2f}")


TRAINED — Medical: 172.34, General: 861.61


In [22]:
# BASE model
base_med = compute_perplexity_mlx(base_model, base_tokenizer, medical_texts)
base_gen = compute_perplexity_mlx(base_model, base_tokenizer, general_texts)
print(f"BASE — Medical: {base_med:.2f}, General: {base_gen:.2f}")


BASE — Medical: 1087081.57, General: 523134.85
